In [33]:
import numpy as np
import psutil
import subprocess
import random as rd
import pickle
import gc
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.mappers import LogarithmicMapper

def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")


def single_line (lines):
    return ''.join(lines.splitlines())

In [34]:
n_x = 6
n_y = 1
n_site = n_x * n_y
n_qubit = n_site
dim = 2**n_qubit

n_qubit_circuit = n_qubit + 1

n_dimer = n_site//2

core  = 0
cores = 1

In [35]:
seed_transpiler = 42

In [36]:
from qiskit import QuantumRegister, QuantumCircuit
qr_circuit = QuantumRegister(n_qubit_circuit,name='q')
#qr = qr_circuit[1:]
#qm = qr_circuit[0]
#indx_qm = 0
indx_qm = n_qubit//2
qm = qr_circuit[indx_qm]
qr = qr_circuit[:indx_qm] + qr_circuit[indx_qm+1:]
#print(qm,qr)

In [37]:
J                   = 1.0
Delta               = -1.0
h                   = 0.0

In [38]:
n_inter             = 2
t_inter_max         = 1.0
t_inters            = [0.0, 1.0]
hamiltonians        = []
mapper = LogarithmicMapper()
for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    hamiltonian_list = []
    # intra-dimer terms
    for i in range(0,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*Delta/4)
        hamiltonian_list.append(term)
        term = ('Z',[ii],-h/2)
        hamiltonian_list.append(term)
        term = ('Z',[jj],-h/2)
        hamiltonian_list.append(term)
    # inter-dimer terms
    for i in range(1,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*t_inter*Delta/4)
        hamiltonian_list.append(term)
    print(hamiltonian_list)
    hamiltonian = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonians.append(hamiltonian.simplify())

    if (core==0):
        print(t_inter, single_line(str(hamiltonians[i_inter])))
        print('')

[('XX', [0, 1], -0.25), ('YY', [0, 1], -0.25), ('ZZ', [0, 1], 0.25), ('Z', [0], -0.0), ('Z', [1], -0.0), ('XX', [2, 3], -0.25), ('YY', [2, 3], -0.25), ('ZZ', [2, 3], 0.25), ('Z', [2], -0.0), ('Z', [3], -0.0), ('XX', [4, 5], -0.25), ('YY', [4, 5], -0.25), ('ZZ', [4, 5], 0.25), ('Z', [4], -0.0), ('Z', [5], -0.0), ('XX', [1, 2], -0.0), ('YY', [1, 2], -0.0), ('ZZ', [1, 2], 0.0), ('XX', [3, 4], -0.0), ('YY', [3, 4], -0.0), ('ZZ', [3, 4], 0.0)]
0.0 SparsePauliOp(['IIIIXX', 'IIIIYY', 'IIIIZZ', 'IIXXII', 'IIYYII', 'IIZZII', 'XXIIII', 'YYIIII', 'ZZIIII'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])

[('XX', [0, 1], -0.25), ('YY', [0, 1], -0.25), ('ZZ', [0, 1], 0.25), ('Z', [0], -0.0), ('Z', [1], -0.0), ('XX', [2, 3], -0.25), ('YY', [2, 3], -0.25), ('ZZ', [2, 3], 0.25), ('Z', [2], -0.0), ('Z', [3], -0.0), ('XX', [4, 5], -0.25), ('YY', [4, 5], -0.25), ('ZZ', [4, 5], 0.25), ('Z', [4], -0.0), ('Z', [5], -0.0), ('XX', [1, 2

In [39]:
init_circuit = QuantumCircuit(qr_circuit)
# init circuit for staring from dimers
n_dimer_half = n_dimer//2
index_attached_to_qm = n_qubit//2-1

#  geometry ; qm -- qr[n_qubit//2-1] -- ... -- qr[1] -- qr[0]
#                   \--qr[n_qubit//2] -- ... -- qr[n_qubit-1]
# apply cxs first
init_circuit.cx(qm,qr[index_attached_to_qm])
for i_qubit in range(index_attached_to_qm,1,-1):
    init_circuit.cx(qr[i_qubit],qr[i_qubit-1])
for i_qubit in range(index_attached_to_qm,n_qubit-2):
    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])


# apply chs + z + cx
for i_dimer in range(n_dimer_half,1,-1):
    i_qubit = 2*i_dimer-1
    init_circuit.ch(qr[i_qubit],qr[i_qubit+1])
    init_circuit.z(qr[i_qubit+1])
    init_circuit.cx(qr[i_qubit+1],qr[i_qubit+2])

for i_dimer in range(n_dimer_half+1,n_dimer):
    i_qubit = 2*i_dimer-1
    init_circuit.ch(qr[i_qubit+1],qr[i_qubit])
    init_circuit.z(qr[i_qubit])
    init_circuit.cx(qr[i_qubit],qr[i_qubit-1])
# boundaries
init_circuit.ch(qr[1],qr[0])
init_circuit.cx(qr[0],qr[1])

init_circuit.ch(qr[-2],qr[-1])
init_circuit.cx(qr[-1],qr[-2])

#
#for i_dimer in range(1,n_dimer_half+1):
#    i_qubit = 2*(i_dimer+n_dimer_half)
#    init_circuit.s(qr[i_qubit+1])
#    init_circuit.h(qr[i_qubit+1])
#    init_circuit.t(qr[i_qubit+1])
#    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])
#    init_circuit.tdg(qr[i_qubit+1])
#    if (i_dimer<n_dimer_half):
#        init_circuit.sx(qr[i_qubit+1])
#        init_circuit.cx(qr[i_qubit+1],qr[i_qubit+2])
#        init_circuit.x(qr[i_qubit+1])
#        init_circuit.h(qr[i_qubit+1])
#    else:
#        init_circuit.h(qr[i_qubit+1])
#        init_circuit.sdg(qr[i_qubit+1])
#    init_circuit.cx(qr[i_qubit+1],qr[i_qubit])




print(init_circuit)
init_circuit_inv = init_circuit.inverse()
print(init_circuit_inv)

               ┌───┐                    
q_0: ──────────┤ H ├──■─────────────────
          ┌───┐└─┬─┘┌─┴─┐               
q_1: ─────┤ X ├──■──┤ X ├───────────────
     ┌───┐└─┬─┘     └───┘          ┌───┐
q_2: ┤ X ├──■────■─────────────────┤ X ├
     └─┬─┘       │                 └─┬─┘
q_3: ──■─────────┼───────────────────┼──
               ┌─┴─┐     ┌───┐┌───┐  │  
q_4: ──────────┤ X ├──■──┤ H ├┤ Z ├──■──
               └───┘┌─┴─┐└─┬─┘└───┘┌───┐
q_5: ───────────────┤ X ├──■────■──┤ X ├
                    └───┘     ┌─┴─┐└─┬─┘
q_6: ─────────────────────────┤ H ├──■──
                              └───┘     
          ┌───┐                         
q_0: ──■──┤ H ├─────────────────────────
     ┌─┴─┐└─┬─┘               ┌───┐     
q_1: ┤ X ├──■─────────────────┤ X ├─────
     ├───┤                    └─┬─┘┌───┐
q_2: ┤ X ├─────────────────■────■──┤ X ├
     └─┬─┘                 │       └─┬─┘
q_3: ──┼───────────────────┼─────────■──
       │  ┌───┐┌───┐     ┌─┴─┐          
q_4: ──■──┤ Z ├┤

In [40]:
n_hamiltonians = len(hamiltonians)
if (core==0):
    print('# Hamiltonian differences')
hamiltonian_diffs = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs.append((hamiltonians[alpha+1]-hamiltonians[alpha]).simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs[alpha])))

if (core==0):
    print('# Hamiltonian differences_list')
hamiltonian_diffs_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_list[alpha])

if (core==0):
    print('# Reduced Hamiltonian differences_list')
hamiltonian_diffs_reduced = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_list = []
    factor_XX = 2 * 2 # 1 for XX, 1 for YY, 2 for symmetric two bonds
    # for 6 site
    ii = index_attached_to_qm
    jj = index_attached_to_qm-1
    term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*factor_XX)
    hamiltonian_list.append(term)
    factor_ZZ = 2 # 2 for symmetric two bonds
    term = ('ZZ',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])*Delta/4*factor_ZZ)
    hamiltonian_list.append(term)
    print(hamiltonian_list)
    dH = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonian_diffs_reduced.append(dH.simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs_reduced[alpha])))
if (core==0):
    print('# Hamiltonian differences_reduced_list')
hamiltonian_diffs_reduced_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_reduced_list.append(hamiltonian_diffs_reduced[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_reduced_list[alpha])

# Hamiltonian differences
0 SparsePauliOp(['IIIXXI', 'IIIYYI', 'IIIZZI', 'IXXIII', 'IYYIII', 'IZZIII'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])
# Hamiltonian differences_list
[('IIIXXI', (-0.25+0j)), ('IIIYYI', (-0.25+0j)), ('IIIZZI', (0.25+0j)), ('IXXIII', (-0.25+0j)), ('IYYIII', (-0.25+0j)), ('IZZIII', (0.25+0j))]
# Reduced Hamiltonian differences_list
[('XX', [2, 1], -1.0), ('ZZ', [2, 1], 0.5)]
0 SparsePauliOp(['IIIXXI', 'IIIZZI'],              coeffs=[-1. +0.j,  0.5+0.j])
# Hamiltonian differences_reduced_list
[('IIIXXI', (-1+0j)), ('IIIZZI', (0.5+0j))]


In [41]:
nmc = 300
beta = 2.0
n_shot = 4096 # this number is not used in real machine calculation,
              # 4096 is just a default number of shots in ibm machines.

n_obs = 3
# 0; norm, 1; dE1, 2; dE2
O_timelists         = [[[[] for _ in range(nmc)] for _ in range(n_obs)] for _ in range(n_hamiltonians)]

exact_dir = '../exact'
with open(exact_dir+'/XXZ6.time.binary','rb') as file_:
    [O_timelists] = pickle.load(file_)

In [42]:
# read exact values
nsec = n_qubit//2 + 1
exact_dir = '../exact'
norms_exact = np.zeros((nsec,n_hamiltonians), dtype=float)
eigen_energies_exact = np.zeros((nsec,n_hamiltonians),dtype=float)
with open(exact_dir + '/exact.norm.E.save','r') as file_:
    alpha = 0
    for line in file_:
        if line.startswith('#'):
            continue
        ls = line.split()
        for isec in range(nsec):
            norms_exact[isec,alpha] = float(ls[isec+1])
            eigen_energies_exact[isec,alpha] = float(ls[isec+nsec+1])
        alpha += 1
for isec in range(nsec):
    for alpha in range(n_hamiltonians):
        print(isec,alpha,norms_exact[isec,alpha],eigen_energies_exact[isec,alpha])

0 0 1.0 0.75
0 1 1.0 1.25
1 0 1.0 -0.25
1 1 0.15550211698203645 -0.6160254037844385
2 0 1.0 -1.25
2 1 0.19329015903038524 -2.0019953568985325
3 0 1.0 -2.2500000000000004
3 1 0.8502049469634537 -2.4935771338879253


In [43]:
def NumberOfTrotterSteps(alpha):
    return 2
if (core==0):
    print('# of Trotter Steps for each alpha')
    for alpha_ in range(1,n_hamiltonians):
        print(NumberOfTrotterSteps(alpha_))

# of Trotter Steps for each alpha
2


In [44]:
from qiskit_ibm_runtime import QiskitRuntimeService
# 
## Load saved credentials
service = QiskitRuntimeService()
backend_name = "ibm_torino" 
backend = service.backend(name=backend_name)

max_circuits = backend.max_circuits

In [45]:
print(max_circuits)

300


In [46]:
initial_layout = [95,96,97,110,98,99,92] # seems to best at 241111, 08:12 PM

In [47]:
eigen_energies_ref = np.zeros((n_hamiltonians),dtype=float)

for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    eigen_energies_ref[i_inter] = - J/4 * Delta * (n_dimer)
    # inter-dimer energies
    eigen_energies_ref[i_inter] += - J/4 * Delta * t_inter* (n_dimer-1) # open boundary condition

In [48]:
print(eigen_energies_ref)

[0.75 1.25]


In [49]:
e_dimer = -0.25 * J * (2.0-Delta)
print (e_dimer)

# read 4 qubit QZMC values
four_qubit_dir = '../../4/ibm_torino'
n_alpha_4 = 1
e_qzmc_4_qubit = 0.0
with open(four_qubit_dir + '/qzmc.norm.E.save','r') as file_:
    alpha = 0
    for line in file_:
        if line.startswith('#'):
            continue
        ls = line.split()
        if (alpha==n_alpha_4):
            e_qzmc_4_qubit = float(ls[-1])
        alpha += 1
print(e_qzmc_4_qubit)

-0.75
-1.6317973385544413


In [50]:
norms_qzmc              = np.ones((n_hamiltonians),dtype=float)
eigen_energies_qzmc     = np.zeros((n_hamiltonians),dtype=float)


eigen_energies_qzmc[0]  = e_dimer*n_dimer
print(eigen_energies_qzmc[0])

-2.25


In [51]:
pauli_identity = 'I'*n_qubit

In [52]:
import time as time_lib

def accumulate_job_results (service_, job_ids_):
    list_out = []
    for job_id_ in job_ids_:
        job_ = service_.job(job_id_)
        while job_.status() != 'DONE':
            time_lib.sleep(30) 
        job_result = job_.result()
        len_result = len(job_result)
        for i in range(len_result):
            list_out.append(job_result[i].data.evs)
    return list_out


In [53]:
from datetime import datetime

In [54]:
job_ids_save = [None for _ in range(n_hamiltonians)]
num_jobs_save = [None for _ in range(n_hamiltonians)]

In [55]:
with open('XXZ6.job_ids','r') as file_:
    lines = file_.readlines()
    ind_line = 0
    for alpha in range(1,n_hamiltonians):
        line = lines[ind_line]
        job_ids = (line.split())
        job_ids_save[alpha]=job_ids
        num_jobs_save[alpha] = len(job_ids)
        #print(job_ids_save[i][alpha])
        ind_line += 1

In [56]:
print(job_ids_save)

[None, ['cwryz3w997wg008xg9dg', 'cwryz6c60bqg008p9rmg', 'cwryz8n5v39g008h94wg', 'cwryzan5v39g008h94x0', 'cwryzcx5v39g008h94xg', 'cwryzfd2ac5g008hzdh0', 'cwryzhe5v39g008h94y0', 'cwryzk6997wg008xg9eg']]


In [57]:
#job = service.job(job_ids_save[1][0])

In [58]:
#job_result = job.result()

In [59]:
#print(job.status()=='DONE')
#for i in range(len(job_result)):
#    print(job_result[i].data.evs)

In [60]:
nhd = len(hamiltonian_diffs_reduced_list[0])

In [61]:
print(nhd)

2


In [62]:
eps = e_dimer+e_qzmc_4_qubit
print(eigen_energies_exact[-1,alpha]-eps)

-0.11177979533348426


In [63]:
# eps = eigen_energies_qzmc[0]
# first-order perturbation correction = 0
# This is better estimate of the energy
# eps = e_2 + e_4
eps = e_dimer + e_qzmc_4_qubit

#comm.Barrier()
for alpha in range(1,n_hamiltonians):
    start_time = datetime.now()
    result = accumulate_job_results(service, job_ids_save[alpha])

    gc.collect()

    pubs = []

    n_pubs = nhd * nmc * 2 * 2 # 2 for indx, 2 for norm
    n_pubs_for_ = [0 for _ in range(cores)]
    remainder         = n_pubs%cores
    for i_core in range(cores):
        n_pubs_for_[i_core] = n_pubs//cores
        if (i_core<remainder):
            n_pubs_for_[i_core] += 1
    if (core==0 and alpha==1):
        print('# of different quantum circuits to run = ', n_pubs)

    result_values_core = [0.0 for _ in range(n_pubs_for_[core])]

    for i in range(n_pubs_for_[core]):
        result_values_core[i] = result[i] # no additional shot errors for real-hardware simulation
    del result

    ### bcast
    #comm.Barrier()
    result_values = []
    for i_core in range(cores):
        if (n_pubs_for_[i_core]==0):
            continue
        result_values_temp = result_values_core
        result_values += result_values_temp
        #comm.Barrier()
    print(result_values)

    norms_1    = np.zeros((nhd,2),dtype=float)
    pauli_norms_1  = np.zeros((nhd,2),dtype=float)
    i_meas = 0
    for ihd in range(nhd):
        # norm, indx = 0, 1
        for indx in range(2):
            for imc in range(nmc):
                norms_1[ihd,indx] += result_values[i_meas]
                i_meas += 1
        # pauli, indx = 0, 1
        for indx in range(2):
            for imc in range(nmc):
                pauli_norms_1[ihd,indx] += result_values[i_meas]
                i_meas += 1
    norms_1 /= nmc
    pauli_norms_1 /= nmc

    print(norms_1)
    print(pauli_norms_1)
    norm_1 = np.sum(norms_1)/(nhd*2)

    # dE1
    dE1 = 0.0
    for ihd in range(nhd):
        norm_avg = 0.0
        for indx in range(2):
            norm_avg += norms_1[ihd,indx]
        pauli_avg = 0.0
        for indx in range(2):
            pauli_avg += pauli_norms_1[ihd,indx]
        pauli_avg /= norm_avg
        coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
        dE1 += coeff * pauli_avg
    dE1 = dE1.real

    # dE1 for each index
    dE1_indx = [0.0, 0.0]
    for ihd in range(nhd):
        coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
        for indx in range(2):
            #val = pauli_norms_1[ihd,indx]/norms_1[ihd,indx]
            # use of norm CXX
            val = pauli_norms_1[ihd,indx]/norms_1[1,indx]
            dE1_indx[indx] += coeff * val
    dE1_indx = np.real(dE1_indx)
    
    #eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1
    # use indx==1 dE for consistency
    eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1_indx[1]
    norms_qzmc[alpha] = norm_1
    del result_values[:]
    del result_values_core[:]
    gc.collect()

    end_time = datetime.now()
    elapsed = end_time-start_time
    elapsed = elapsed.total_seconds()
    if (core==0):
        st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
        memory_usage(st)
        print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[-1,alpha])
        print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[0]+dE1_indx[0]-eigen_energies_exact[-1,alpha])
        print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[0]+dE1_indx[1]-eigen_energies_exact[-1,alpha])
        if (alpha<n_hamiltonians-1):
            print('precision of the predictor for next', eps-eigen_energies_exact[-1,alpha+1])
        st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
        print(st)



# of different quantum circuits to run =  2400
[array(0.58559919), array(0.54380665), array(0.51107754), array(0.50352467), array(0.48992951), array(0.5347432), array(0.53927492), array(0.53524673), array(0.54229607), array(0.50906344), array(0.46978852), array(0.53524673), array(0.63141994), array(0.56193353), array(0.57452165), array(0.5060423), array(0.49748238), array(0.51007049), array(0.59365559), array(0.49748238), array(0.63846928), array(0.53021148), array(0.50352467), array(0.59365559), array(0.59869084), array(0.47280967), array(0.48942598), array(0.63141994), array(0.49446123), array(-0.00050352), array(0.6570997), array(0.63997986), array(0.5060423), array(0.59718026), array(0.55085599), array(0.50453172), array(0.58056395), array(0.4939577), array(0.51007049), array(0.57854985), array(0.60976838), array(0.36908359), array(0.61127895), array(0.59466264), array(0.4959718), array(0.57049345), array(0.52668681), array(0.62437059), array(-0.14400806), array(0.65458207), array(

In [32]:
# save to file
with open('qzmc.norm.E.save','w') as file_:
    s = '# t_inter, norms, E'
    s += '\n'
    file_.write(s)
    for alpha in range(n_inter):
        t_inter = t_inters[alpha]
        s = '{:}'.format(t_inter)
        s += '  {:.16e}'.format(norms_qzmc[alpha])
        s += '  {:.16e}'.format(eigen_energies_qzmc[alpha])
        print(s)
        s += '\n'
        file_.write(s)

0.0  1.0000000000000000e+00  -2.2500000000000000e+00
1.0  4.9501007448537321e-01  -2.4681834492930093e+00
